# Интерпретация

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import partial_dependence, PartialDependenceDisplay, permutation_importance
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import lime
import lime.lime_tabular
from torch.utils.hipify.hipify_python import preprocessor
from torchmetrics.functional import mean_absolute_percentage_error

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'retina'


### Загрузка и подготовка данных


Вам будет предоставлен датасет, посвященный продаже недвижимости. Ваша задача - построить интерпретацию для этого датасета. В нем достаточно много различных признаков, поэтому вы можете предварительно отфильтровать их, когда будете строить графики. Оставляйте достаточно признаков, чтобы ваши модели оставались точными..

In [ ]:
data_path = 'hw_interpretation/data.csv'
data = pd.read_csv(data_path, sep=',')

print(f"Размер датасета: {data.shape}")
print(f"\nПервые строки:")
data.head()

## Задание 1. 1 балл
Сделайте 2 версии данных - с нормализацией признаков и без.
Обучите 6 моделей:
- линейную регрессию (LinearRegression) на двух вариантах данных
- Lasso регрессию (Lasso) на двух вариантах данных
- градиентный бустинг (GradientBoostingRegressor) на двух вариантах данных. Ограничьте глубину до 5.

Выведите MSE,RMSE и MAPE моделей. Какая функция больше подходит? Почему?

Зафиксируйте выводы. Какие модели чувствительны к масштабу признаков, а какие почти инвариантны? Почему это важно для анализа признаков?

1) Подготовка данных

In [ ]:
missing_stats = pd.DataFrame({
    'Колонка': data.columns,
    'Тип': data.dtypes.values,
    'Пропуски': data.isnull().sum().values,
    'Процент': (data.isnull().sum() / len(data) * 100).values
})
missing_stats = missing_stats[missing_stats['Пропуски'] > 0].sort_values('Пропуски', ascending=False)
print(missing_stats.to_string(index=False))

In [ ]:
def process_date(df, date_column):
    df = df.copy() 
    df[date_column] = pd.to_datetime(df[date_column])
    df["year"] = df[date_column].dt.year
    df["month"] = df[date_column].dt.month
    df["day"] = df[date_column].dt.day
    df = df.drop(columns=[date_column])
    return df


data = process_date(data, date_column='agreement_date')

In [ ]:
def clean_rooms(value):
    if pd.isna(value):  
        return np.nan
    value = str(value).strip().lower()  
    if value == 'студия':
        return 0
    if value == '>=4':
        return 4
    if value == 'rooms_4':
        return np.nan  
    try:
        return float(value)
    except:
        return np.nan 
    
data['rooms_4'] = data['rooms_4'].apply(clean_rooms)
median_rooms = data['rooms_4'].median()
data['rooms_4'].fillna(median_rooms, inplace=True)

In [ ]:
def replace_unset_with_nan(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].replace('<unset>', np.nan)
    return df


data = replace_unset_with_nan(data)

2) Разделение на нормализованые и ненормализованые данные 

In [ ]:
X, y = data.drop(columns=["price_target"]), data["price_target"]
data.head()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
categorical_cols = ['region_name_cat', 'hc_name_cat', 'interior_cat', 'class_cat', 'stage_cat']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor_not_scaled = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  
            ('passthrough', 'passthrough')
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),  
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ])
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ])


X_train_not_scaled, X_test_not_scaled = preprocessor_not_scaled.fit_transform(X_train), preprocessor_not_scaled.transform(X_test)

X_train_scaled, X_test_scaled = preprocessor_scaled.fit_transform(X_train), preprocessor_scaled.transform(X_test)

3) Оценка

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_test, pred) * 100 
    return mse, rmse, mape

4) Модели и их вывод

In [ ]:
lr_raw = evaluate(LinearRegression(), X_train_not_scaled, y_train, X_test_not_scaled, y_test)
lr_scaled = evaluate(LinearRegression(), X_train_scaled, y_train, X_test_scaled, y_test)

lasso_raw = evaluate(Lasso(alpha=1.0), X_train_not_scaled, y_train, X_test_not_scaled, y_test)
lasso_scaled = evaluate(Lasso(alpha=1.0), X_train_scaled, y_train, X_test_scaled, y_test)

gb_raw = evaluate(GradientBoostingRegressor(max_depth=5, random_state=42), X_train_not_scaled, y_train, X_test_not_scaled, y_test)
gb_scaled = evaluate(GradientBoostingRegressor(max_depth=5, random_state=42), X_train_scaled, y_train, X_test_scaled,
                     y_test)

In [ ]:
models = {
    "Lin Not Scaled": lr_raw, 
    "Lin Scaled": lr_scaled,
    "Lasso Not Scaled": lasso_raw, 
    "Lasso Scaled": lasso_scaled,
    "Gradient Boost Not Scaled": gb_raw, 
    "Gradient Boost Scaled": gb_scaled
}

print(f"{'Модель':<20} {'MSE':>12} {'RMSE':>12} {'MAPE(%)':>12}")
for name, (mse, rmse, mape) in models.items():
    print(f"{name:<20} {mse:>12.2f} {rmse:>12.2f} {mape:>12.2f}")

In [ ]:
print("Анализ чувствительности к нормированию")
model_pairs = [('Lin', 'Lin Not Scaled', 'Lin Scaled'),
               ('Lasso', 'Lasso Not Scaled', 'Lasso Scaled'),
               ('Gradient Boost', 'Gradient Boost Not Scaled', 'Gradient Boost Scaled')]

for name, not_scaled, scaled in model_pairs:
    rmse_not_scaled = models[not_scaled][1]
    rmse_scaled = models[scaled][1]
    diff = np.abs(rmse_not_scaled - rmse_scaled) / rmse_not_scaled * 100
    status = "чувствительна" if diff > 1 else "устойчива"
    
    print(f"\n{name}: {status} (разница {diff:.2f}%)")
    print(f"  без нормир.: RMSE={rmse_not_scaled:.2f}, MAPE={models[not_scaled][2]:.2f}%")
    print(f"  с нормир.:   RMSE={rmse_scaled:.2f}, MAPE={models[scaled][2]:.2f}%")

## Задание 1.1(*) 1 балл
Сравните модели, построенные с помощью разных видов нормализации (MinMax, Standart). Отличается ли важность признаков?

In [ ]:
from sklearn.preprocessing import MinMaxScaler

preprocessor_minmax = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  
            ('scaler', MinMaxScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ])

X_train_minmax = preprocessor_minmax.fit_transform(X_train)
X_test_minmax = preprocessor_minmax.transform(X_test)

In [ ]:
lr_minmax = evaluate(LinearRegression(), X_train_minmax, y_train, X_test_minmax, y_test)
lasso_minmax = evaluate(Lasso(alpha=1.0), X_train_minmax, y_train, X_test_minmax, y_test)
gb_minmax = evaluate(GradientBoostingRegressor(max_depth=5, random_state=42), 
                     X_train_minmax, y_train, X_test_minmax, y_test)

In [ ]:
models.update({
    "Lin MinMax": lr_minmax,
    "Lasso MinMax": lasso_minmax,
    "Gradient Boost MinMax": gb_minmax
})

In [ ]:
print(f"{'Модель':<25} {'RMSE':>12} {'MAPE(%)':>12}")

for model in ['Lin', 'Lasso', 'Gradient Boost']:
    not_scaled = models[f"{model} Not Scaled"][1]
    standard = models[f"{model} Scaled"][1]
    minmax = models[f"{model} MinMax"][1]
    
    print(f"\n{model}:")
    print(f"  Not Scaled:  RMSE={not_scaled:.2f}")
    print(f"  Standard:     RMSE={standard:.2f} (разница {np.abs(standard-not_scaled)/not_scaled*100:.2f}%)")
    print(f"  MinMax:       RMSE={minmax:.2f} (разница {np.abs(minmax-not_scaled)/not_scaled*100:.2f}%)")


In [ ]:
cat_features = preprocessor_not_scaled.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
feature_names = list(numerical_cols) + list(cat_features)

print("\nLasso - Топ-5 признаков по коэффициентам:")

lasso_models = {
    'Standard': Lasso(alpha=1.0).fit(X_train_scaled, y_train),
    'MinMax': Lasso(alpha=1.0).fit(X_train_minmax, y_train)
}

for name, model in lasso_models.items():
    coefs = model.coef_
    top_idx = np.argsort(np.abs(coefs))[::-1][:5]
    print(f"\n  {name}:")
    for i, idx in enumerate(top_idx):
        if np.abs(coefs[idx]) > 0.0001:
            print(f"    {i+1}. {feature_names[idx]}: {coefs[idx]:.4f}")

In [ ]:
print("\nGradient Boosting - Топ-5 признаков по важности:")

gb_models = {
    'Not Scaled': GradientBoostingRegressor(max_depth=5, random_state=42).fit(X_train_not_scaled, y_train),
    'Standard': GradientBoostingRegressor(max_depth=5, random_state=42).fit(X_train_scaled, y_train),
    'MinMax': GradientBoostingRegressor(max_depth=5, random_state=42).fit(X_train_minmax, y_train)
}

for name, model in gb_models.items():
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:5]
    print(f"\n  {name}:")
    for i, idx in enumerate(top_idx):
        print(f"    {i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

## Задание 2. 1 балл
Выберите 1 признак для анализа (можно категориальный, с не менее чем 5 уровнями, или дискретизируйте непрерывный). 
Используйте линейную регрессию и бустинг после применения MinMaxScaler. Что будет с моделями, если признаки выйдут из диапазона?
Постройте графики ICE и PDP для интерпретации исходных данных, а также искусственно добавив несколько выбросов, выходящих за оригинальные интервалы. 

Задание 2.1 (*) 1 балл: проанализируйте также еще один признак

In [ ]:
feature_1 = 'floor'
feature_2 = 'square'

In [ ]:
def train_models_minmax(X_train, y_train, X_test, y_test, preprocessor_minmax):
    X_train_mm = preprocessor_minmax.fit_transform(X_train)
    X_test_mm = preprocessor_minmax.transform(X_test)
    
    models = {
        "Linear": LinearRegression(),
        "Boosting": GradientBoostingRegressor(max_depth=5, random_state=42)
    }
    
    trained = {}
    
    for name, model in models.items():
        model.fit(X_train_mm, y_train)
        trained[name] = model
        
    return trained, X_train_mm, X_test_mm

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

def plot_pdp_ice(models, X, feature_index, feature_name):
    for name, model in models.items():
        print(f"\n{name} - PDP + ICE для {feature_name}")
        
        PartialDependenceDisplay.from_estimator(
            model,
            X,
            [feature_index],
            kind="both",  # PDP + ICE
            subsample=1000,
            random_state=42
        )

In [ ]:
def get_feature_index(feature_name, numerical_cols, categorical_cols, preprocessor):
    cat_features = preprocessor.named_transformers_['cat'] \
        .named_steps['onehot'].get_feature_names_out(categorical_cols)
    
    feature_names = list(numerical_cols) + list(cat_features)
    
    return feature_names.index(feature_name), feature_names

In [ ]:
def add_outliers(X, feature_name, factor=2):
    X_out = X.copy()
    
    col_idx = X.columns.get_loc(feature_name)
    
    max_val = X.iloc[:, col_idx].max()
    
    outliers = X.sample(50, random_state=42).copy()
    outliers.iloc[:, col_idx] = max_val * factor
    
    X_augmented = pd.concat([X, outliers], axis=0)
    
    return X_augmented

In [ ]:
models, X_train_mm, X_test_mm = train_models_minmax(
    X_train, y_train, X_test, y_test, preprocessor_minmax
)

In [ ]:
feature_idx, feature_names = get_feature_index(
    feature_1,
    numerical_cols,
    categorical_cols,
    preprocessor_minmax
)

In [ ]:
plot_pdp_ice(models, X_train_mm, feature_idx, feature_1)

In [ ]:
X_train_outliers = add_outliers(X_train, feature_1)

X_train_outliers_mm = preprocessor_minmax.transform(X_train_outliers)

In [ ]:
plot_pdp_ice(models, X_train_outliers_mm, feature_idx, feature_1 + " (outliers)")

In [ ]:
feature_idx, _ = get_feature_index(
    feature_2,
    numerical_cols,
    categorical_cols,
    preprocessor_minmax
)

plot_pdp_ice(models, X_train_mm, feature_idx, feature_2)

X_train_outliers = add_outliers(X_train, feature_2)
X_train_outliers_mm = preprocessor_minmax.transform(X_train_outliers)

plot_pdp_ice(models, X_train_outliers_mm, feature_idx, feature_2 + " (outliers)")


## Задание 3. 1 балл
Выберите 20 объектов из тестовой выборки.
Для каждого объекта из выбранного набора построим траекторию изменения предсказания модели при постепенном изменении значения признака от его текущего значения к базовому значению (медиана или среднее по обучающей выборке).

**Алгоритм:**
1. Выбрать объект $x_i$ из тестовой выборки
2. Для интересующего признака $j$:
   - Текущее значение: $x_{i,j}$
   - Базовое значение: $x_{base,j}$ (медиана или среднее по обучающей выборке)
3. Построить линейную интерполяцию между $x_{i,j}$ и $x_{base,j}$ с $n$ шагами
4. Для каждого шага интерполяции:
   - Заменить значение признака $j$ в объекте $x_i$ на значение из интерполяции
   - Вычислить предсказание модели для модифицированного объекта
5. Построить график траектории


Задание 3.1 (*) 1 балл: проанализируйте также еще один признак

In [ ]:
def get_base_value(X_train, feature_name, mode='median'):
    if mode == 'median':
        return X_train[feature_name].median()
    elif mode == 'mean':
        return X_train[feature_name].mean()
    else:
        raise ValueError("mode должен быть 'median' или 'mean'")

In [ ]:
def interpolate_values(start, end, n_steps=20):
    return np.linspace(start, end, n_steps)

In [ ]:
def compute_trajectory(model, preprocessor, x_row, feature_name, base_value, n_steps=20):
    x_current = x_row.copy()
    
    start_value = x_current[feature_name]
    values = interpolate_values(start_value, base_value, n_steps)
    
    preds = []
    
    for v in values:
        x_temp = x_current.copy()
        x_temp[feature_name] = v
        
        x_transformed = preprocessor.transform(pd.DataFrame([x_temp]))
        pred = model.predict(x_transformed)[0]
        
        preds.append(pred)
    
    return values, preds

In [ ]:
import matplotlib.pyplot as plt

def plot_trajectories(models, preprocessor, X_test, X_train, feature_name, n_objects=20, n_steps=20):
    
    sample = X_test.sample(n_objects, random_state=42)
    
    base_value = get_base_value(X_train, feature_name, mode='median')
    
    for model_name, model in models.items():
        plt.figure()
        
        print(f"\n{model_name} — признак: {feature_name}")
        
        for _, row in sample.iterrows():
            values, preds = compute_trajectory(
                model,
                preprocessor,
                row,
                feature_name,
                base_value,
                n_steps
            )
            
            plt.plot(values, preds, alpha=0.5)
        
        plt.axvline(base_value, linestyle='--')
        plt.title(f"{model_name}: trajectory for {feature_name}")
        plt.xlabel(feature_name)
        plt.ylabel("prediction")
        plt.show()

In [ ]:
feature_1 = 'floor'

plot_trajectories(
    models,
    preprocessor_minmax,
    X_test,
    X_train,
    feature_1
)

In [ ]:
feature_2 = 'square'

plot_trajectories(
    models,
    preprocessor_minmax,
    X_test,
    X_train,
    feature_2
)

## Задание 4 (1 балл). ALE
Постройте ALE по обеим моделям, используя pyALE. Подберите размер сетки так, чтобы получить доверительные интервалы. Проанализируйте полученный график. Каковы получились доверительные интервалы? Почему они различны для моделей?

P.s. Сетку значений стройте для исходного признака.

In [ ]:
class ALEModelWrapper:
    def __init__(self, model, preprocessor):
        self.model = model
        self.preprocessor = preprocessor
    
    def predict(self, X):
        X_transformed = self.preprocessor.transform(X)
        return self.model.predict(X_transformed)

In [ ]:
from PyALE import ale

In [ ]:
import matplotlib.pyplot as plt

def plot_ale(models, preprocessor, X_train, feature_name, bins=20):
    
    X_sample = X_train.sample(2000, random_state=42) if len(X_train) > 2000 else X_train
    
    for model_name, model in models.items():
        print(f"\n{model_name} — ALE для {feature_name}")
        
        wrapped_model = ALEModelWrapper(model, preprocessor)
        
        plt.figure()
        
        ale_eff = ale(
            X=X_sample,
            model=wrapped_model, 
            feature=[feature_name],
            grid_size=bins,
            include_CI=True
        )
        
        plt.title(f"{model_name} ALE: {feature_name}")
        plt.show()

In [ ]:
feature_1 = 'floor'

plot_ale(
    models,
    preprocessor_minmax,
    X_train,
    feature_1,
    bins=20
)

In [ ]:
feature_2 = 'square'

plot_ale(
    models,
    preprocessor_minmax,
    X_train,
    feature_2,
    bins=20
)

## Задание 5:  Permutation Importance (2 балла)
Постройте Permutation importances по обеим моделям, используя sklearn.

Поэкспериментируйте с числом перестановок.

Проанализируйте полученные коэффициенты. Как они меняются от количества перестановок? Как меняются std коэффициентов?



In [ ]:
from sklearn.inspection import permutation_importance

def compute_permutation_importance(model, X_test, y_test, feature_names, n_repeats=10):
    
    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1
    )
    
    importances = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std
    }).sort_values("importance_mean", ascending=False)
    
    return importances

In [ ]:
def get_feature_names(preprocessor, numerical_cols, categorical_cols):
    cat_features = preprocessor.named_transformers_['cat'] \
        .named_steps['onehot'].get_feature_names_out(categorical_cols)
    
    return list(numerical_cols) + list(cat_features)

In [ ]:
def permutation_experiment(models, X_test, y_test, feature_names, repeats_list=[5, 10, 30]):
    
    results = {}
    
    for model_name, model in models.items():
        print(model_name)
        
        results[model_name] = {}
        
        for n in repeats_list:
            print(f"\n n_repeats = {n}")
            
            imp = compute_permutation_importance(
                model,
                X_test,
                y_test,
                feature_names,
                n_repeats=n
            )
            
            print(imp.head(5))
            
            results[model_name][n] = imp
    
    return results

In [ ]:
import matplotlib.pyplot as plt

def plot_importance_stability(results, model_name, feature, repeats_list):
    
    means = []
    stds = []
    
    for n in repeats_list:
        df = results[model_name][n]
        row = df[df["feature"] == feature]
        
        means.append(row["importance_mean"].values[0])
        stds.append(row["importance_std"].values[0])
    
    plt.figure()
    
    plt.plot(repeats_list, means, marker='o', label='mean importance')
    plt.plot(repeats_list, stds, marker='o', linestyle='--', label='std')
    
    plt.xlabel("n_repeats")
    plt.ylabel("value")
    plt.title(f"{model_name} — {feature}")
    plt.legend()
    
    plt.show()

In [ ]:
feature_names = get_feature_names(
    preprocessor_minmax,
    numerical_cols,
    categorical_cols
)

In [ ]:
results = permutation_experiment(
    models,
    X_test_mm,
    y_test,
    feature_names,
    repeats_list=[5, 10, 30]
)

In [ ]:
plot_importance_stability(results, "Linear", feature_names[0], [5, 10, 30])
plot_importance_stability(results, "Boosting", feature_names[0], [5, 10, 30])

## Задание 5: Feature Importance (2 балла)
Пусть важность - это MAPE для тестовых данных. Проведите анализ только для бустинга

Идея перестановочной важности представляет собой частный случай важности при помощи внесения возмущений в признак. Примеры возмущений:
1) внесение случайного шума
2) зануление признака
3) сдвиг признака к его базовому значению и оценка траектории изменения прогнозов или качества модели 

Примем за базовое значение (${base}$)медиану признака и будем сдвигать исходный признак к медианному с некоторым коэффициентом $\beta$:
$x_j^\beta = (1- \beta)x_j + \beta {base}$

Реализуйте это возмущение. Как меняются важности при разных $\beta$?

Постройте графики важности и сравните важности с permutation importance. Используйте только числовые признаки. При этом медиану стоит считать на тренировочном наборе, а важность как разницу MAPE на тестовой выборке. Чем больше разница, тем важнее признак. 


Сравните результаты методов. Какие признаки наиболее важны? Есть ли различия между методами? В чём могут быть причины различий?

In [ ]:
numerical_only = numerical_cols.copy()

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

def compute_baseline_mape(model, preprocessor, X_test, y_test):
    X_test_tr = preprocessor.transform(X_test)
    pred = model.predict(X_test_tr)
    return mean_absolute_percentage_error(y_test, pred) * 100

In [ ]:
def perturb_feature(X, feature, base_value, beta):
    X_new = X.copy()
    
    X_new[feature] = (1 - beta) * X_new[feature] + beta * base_value
    
    return X_new

In [ ]:
def compute_feature_importance_beta(model, preprocessor, X_test, y_test, 
                                    feature, base_value, beta, baseline_mape):
    
    X_perturbed = perturb_feature(X_test, feature, base_value, beta)
    
    X_tr = preprocessor.transform(X_perturbed)
    pred = model.predict(X_tr)
    
    mape = mean_absolute_percentage_error(y_test, pred) * 100
    
    return mape - baseline_mape

In [ ]:
def compute_importance_for_all_features(model, preprocessor, X_train, X_test, y_test, 
                                        numerical_features, beta):
    
    baseline_mape = compute_baseline_mape(model, preprocessor, X_test, y_test)
    
    importances = []
    
    for feature in numerical_features:
        base_value = X_train[feature].median()
        
        imp = compute_feature_importance_beta(
            model,
            preprocessor,
            X_test,
            y_test,
            feature,
            base_value,
            beta,
            baseline_mape
        )
        
        importances.append((feature, imp))
    
    df = pd.DataFrame(importances, columns=["feature", "importance"])
    df = df.sort_values("importance", ascending=False)
    
    return df

In [ ]:
def beta_experiment(model, preprocessor, X_train, X_test, y_test, 
                    numerical_features, betas=[0.25, 0.5, 0.75, 1.0]):
    
    results = {}
    
    for beta in betas:
        print(f"beta = {beta}")
        
        df = compute_importance_for_all_features(
            model,
            preprocessor,
            X_train,
            X_test,
            y_test,
            numerical_features,
            beta
        )
        
        print(df.head(5))
        
        results[beta] = df
    
    return results

In [ ]:
import matplotlib.pyplot as plt

def plot_beta_importance(results, feature):
    
    betas = sorted(results.keys())
    values = []
    
    for b in betas:
        df = results[b]
        val = df[df["feature"] == feature]["importance"].values[0]
        values.append(val)
    
    plt.figure()
    plt.plot(betas, values, marker='o')
    plt.xlabel("beta")
    plt.ylabel("importance (ΔMAPE)")
    plt.title(f"Importance vs beta: {feature}")
    plt.show()

In [ ]:
boosting_model = models["Boosting"]

In [ ]:
results_beta = beta_experiment(
    boosting_model,
    preprocessor_minmax,
    X_train,
    X_test,
    y_test,
    numerical_only
)

In [ ]:
top_feature = results_beta[1.0].iloc[0]["feature"]

plot_beta_importance(results_beta, top_feature)

#  Задание 6. 2 балла. LIME.
Постройте интерпретацию признаков для нескольких примеров с помощью LIME. Можете использовать 
Оцените устойчивость реализации. Как влияет на коэффициенты количество сгенерированных точек? А выбор признаков (lasso/добавление фичей по порядку). А выбор ядра?

(*) Вы получите на 2 балла больше, если используете свою реализацию из задания семинарского ноутбука. В таком случае не забудьте добавить тесты для своей реализации. 



In [ ]:
class LIME:
    def __init__(self, train_data, kernel, kernel_width=3.0, n_samples=5000, random_state=42):
        self.train_data = train_data
        self.kernel = kernel
        self.kernel_width = kernel_width
        self.n_samples = n_samples
        self.random_state = np.random.RandomState(random_state)

        self.binary_mask = (np.unique(train_data, axis=0).shape[0] < 10)  # fallback
        self.binary_mask = np.all((train_data == 0) | (train_data == 1), axis=0)

        self.stds = np.std(train_data, axis=0)

    def _generate_samples(self, instance):
        n_features = len(instance)
        samples = np.zeros((self.n_samples, n_features))

        for j in range(n_features):
            if self.binary_mask[j]:
                p = instance[j]
                samples[:, j] = self.random_state.binomial(1, p, size=self.n_samples)
            else:
                scale = self.stds[j] * 0.2 + 1e-6
                samples[:, j] = self.random_state.normal(
                    loc=instance[j],
                    scale=scale,
                    size=self.n_samples
                )

        return samples

    def explain(self, instance, predict_fn, num_features=5):
        instance = np.asarray(instance).reshape(-1)

        samples = self._generate_samples(instance)

        preds = predict_fn(samples)
        if preds.ndim > 1:
            preds = preds[:, 1]

        distances = np.sqrt(((samples - instance) ** 2).sum(axis=1))

        distances = distances / (np.std(distances) + 1e-8)

        weights = self.kernel(distances, self.kernel_width)
        weights = weights / (weights.mean() + 1e-8)

        X = samples
        y = preds

        W = np.sqrt(weights)
        X_w = X * W[:, None]
        y_w = y * W

        model = Lasso(alpha=1e-6, max_iter=10000)
        model.fit(X_w, y_w)

        coefs = model.coef_

        top_idx = np.argsort(-np.abs(coefs))[:num_features]

        explanation = {
            f"feature_{i}": coefs[i]
            for i in top_idx
        }

        return explanation, model

In [ ]:
def kernel(d, kernel_width):
    return np.exp(-(d ** 2) / (kernel_width ** 2))

In [ ]:
X_train_lime = X_train_mm
X_test_lime = X_test_mm

In [ ]:
def predict_fn_lime(X):
    return boosting_model.predict(X)

In [ ]:
feature_names = get_feature_names(
    preprocessor_minmax,
    numerical_cols,
    categorical_cols
)

In [ ]:
lime = LIME(X_train_lime, kernel)

for i in range(3):
    instance = X_test_lime[i]
    
    explanation, _ = lime.explain(instance, predict_fn_lime)
    
    explanation_named = {
        feature_names[int(k.split('_')[1])]: v
        for k, v in explanation.items()
    }
    
    print(f"Объект {i}:")
    for k, v in explanation_named.items():
        print(f"{k}: {v:.4f}")

Тесты

In [ ]:
def test_lime_runs():
    explanation, _ = lime.explain(X_test_lime[0], predict_fn_lime)
    assert isinstance(explanation, dict)


def test_num_features():
    explanation, _ = lime.explain(X_test_lime[0], predict_fn_lime, num_features=3)
    assert len(explanation) <= 3


def test_nonzero_weights():
    explanation, _ = lime.explain(X_test_lime[0], predict_fn_lime)
    assert len(explanation) > 0  


test_lime_runs()
test_num_features()
test_nonzero_weights()

Влияние n_samples 

In [ ]:
for n in [500, 2000, 5000]:
    lime.n_samples = n
    
    explanation, _ = lime.explain(X_test_lime[0], predict_fn_lime)
    
    print(f"n_samples = {n}")
    print(explanation)

Lasso vs LinearRegression

In [ ]:
from sklearn.linear_model import LinearRegression

def explain_with_model(model_class):
    instance = X_test_lime[0]

    samples = lime._generate_samples(instance)
    preds = predict_fn_lime(samples)

    if preds.ndim > 1:
        preds = preds[:, 1]

    scaler = StandardScaler()
    samples_scaled = scaler.fit_transform(samples)
    instance_scaled = scaler.transform(instance.reshape(1, -1))

    distances = np.sqrt(((samples_scaled - instance_scaled) ** 2).sum(axis=1))
    weights = kernel(distances, lime.kernel_width)

    W = np.sqrt(weights)
    X_w = samples_scaled * W[:, None]
    y_w = preds * W

    model = model_class()
    model.fit(X_w, y_w)

    coefs = model.coef_
    top_idx = np.argsort(-np.abs(coefs))[:5]

    return {f"feature_{i}": coefs[i] for i in top_idx}


print("Lasso:")
print(explain_with_model(lambda: Lasso(alpha=0.001)))

print("LinearRegression:")
print(explain_with_model(LinearRegression))

Влияние kernel_width

In [ ]:
for kw in [0.25, 0.75, 2.0]:
    lime.kernel_width = kw
    
    explanation, _ = lime.explain(X_test_lime[0], predict_fn_lime)
    
    print(f"\nkernel_width = {kw}")
    print(explanation)

## Задание 7. 1 балл. SHAP
Постройте локальный график с SHAP для объекта с индексом, равным вашему номеру в таблице курса на обеих моделях и сделайте выводы. 
## Задание 7.1 (*). 1 балл.  Shap и категориальные переменные.
Shap разлагает предсказание модели вблизи точки x на базовый уровень и сумму вкладов признаков: $ f(x) = base + \sum_i{\phi_i(x)}$. В случае one-hot вклад признака - это сумма вкладов dummy столбцов. Сравните вклады категориальных признаков до и после кодировки - так ли это? 


In [ ]:
import shap

In [ ]:
X_lime = X_test_lime  

In [ ]:
idx = 3
instance = X_lime[idx:idx+1] 

In [ ]:
linear_model = models["Linear"]

In [ ]:
explainer_gb = shap.Explainer(boosting_model)
shap_values_gb = explainer_gb(instance)
shap_values_gb.feature_names = feature_names  

explainer_lr = shap.Explainer(linear_model, X_lime)
shap_values_lr = explainer_lr(instance)
shap_values_lr.feature_names = feature_names 

In [ ]:
shap.initjs()

print("Gradient Boosting")
shap.plots.waterfall(shap_values_gb[0])

print("Linear Regression")
shap.plots.waterfall(shap_values_lr[0])

In [ ]:
feature_names = get_feature_names(
    preprocessor_minmax,
    numerical_cols,
    categorical_cols
)

cat_name = 'hc_name_cat'

mask = [i for i, f in enumerate(feature_names) if cat_name in f]

hc_contribution_gb = shap_values_gb.values[0, mask].sum()
hc_contribution_lr = shap_values_lr.values[0, mask].sum()

print(f"\nВклад {cat_name} в GB: {hc_contribution_gb:.2f}")
print(f"Вклад {cat_name} в LR: {hc_contribution_lr:.2f}")